In [10]:
import pandas as pd
import os
import glob
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import datetime as dt

# =============================================================================
# VARIABLES DE CONFIGURACIÓN
# =============================================================================
#folder_path = r"C:\Users\ricar\OneDrive\Almacen\Trabajo\5. SIMAC\10. Series\2026-09"
folder_path = r"C:\Users\ricar\Downloads\Revision Filtros\Revision 2"
ruta_base_salida = r"C:\Users\ricar\OneDrive\Almacen\Trabajo\5. SIMAC\9. Indicadores_climaticos\4.Revision_de_series\7.Agrupacion de series\4.Resultados"

filas_a_saltar = 3 
niveles_carpetas = 3

variable_a_analizar = "Precipitación [mm]" #Se deben introducir de la siguiente manera: Temperatura [°C],Precipitación [mm],Nivel Corregido [cm], Velocidad del Viento,Dirección del Viento,Dirección de la Rosa,Presión Barométrica [mmHg],Humedad Relativa [%],Radiación Solar [W/m2],Evapotranspiración 

# Define la escala de agrupación deseada. 
# Opciones válidas: 'diario', 'semanal', 'mensual', 'trimestral', 'semestral', 'anual'
escala_agrupacion = 'diario'

# =============================================================================
# VARIABLES DE SELECCIÓN Y LÍMITE DE ARCHIVOS
# =============================================================================
procesar_todos = False          # True: ignora los filtros y procesa todo. False: aplica los filtros.

# Variable para decidir CUÁNTOS archivos analizar (ej. 3). 
# Usa 'None' para apagar esta restricción.
limite_archivos = None          

# Variable para decidir CUÁLES archivos analizar según su posición en la lista.
# Recuerda que en Python el conteo empieza en 0. Ej: [0, 1, 2] lee los tres primeros.
# Deja la lista vacía [] para apagar esta restricción.
indices_seleccionados = []

# =============================================================================
# DICCIONARIO DE ESCALAS Y SUFIJOS
# =============================================================================
# Utiliza las etiquetas actualizadas de pandas (ME, QE, YE) para evitar advertencias
escalas = {
    'diario': {'resample': 'D', 'periodo': 'Diario', 'label_x': 'Día-Mes-Año', 'formato_fecha': '%d-%m-%Y'},
    'semanal': {'resample': 'W', 'periodo': 'Semanal', 'label_x': 'Día-Mes-Año', 'formato_fecha': '%d-%m-%Y'},
    'mensual': {'resample': 'ME', 'periodo': 'Mensual', 'label_x': 'Mes-año', 'formato_fecha': '%m-%Y'},
    'trimestral': {'resample': 'QE', 'periodo': 'Trimestral', 'label_x': 'Trimestre-año', 'formato_fecha': '%m-%Y'},
    'semestral': {'resample': '6ME', 'periodo': 'Semestral', 'label_x': 'Semestre-año', 'formato_fecha': '%m-%Y'},
    'anual': {'resample': 'YE', 'periodo': 'Anual', 'label_x': 'Año', 'formato_fecha': '%Y'}
}

if escala_agrupacion not in escalas:
    raise ValueError("Escala no válida. Las opciones son: diario, semanal, mensual, trimestral, semestral, anual")

config_escala = escalas[escala_agrupacion]
regla_resample = config_escala['resample']
periodo_elegido = config_escala['periodo']
label_eje_x = config_escala['label_x']
formato_fecha = config_escala['formato_fecha']

# =============================================================================
# EXTRACCIÓN Y AGRUPACIÓN DE DATOS
# =============================================================================
csv_files = []
for nivel in range(niveles_carpetas):
    subcarpetas = ["*"] * nivel
    file_pattern = os.path.join(folder_path, *subcarpetas, "*.csv")
    csv_files.extend(glob.glob(file_pattern))

# Es fundamental ordenar la lista para que los índices seleccionados 
# siempre correspondan exactamente a los mismos archivos en cada ejecución.
csv_files = sorted(csv_files)

# --- INICIO DE LÓGICA DE FILTRADO INDEPENDIENTE ---
if not procesar_todos:
    # 1er Filtro: Extraer solo los índices específicos si la lista no está vacía
    if len(indices_seleccionados) > 0:
        csv_files = [csv_files[i] for i in indices_seleccionados if i < len(csv_files)]
        
    # 2do Filtro: Limitar la cantidad total si hay un límite establecido
    if limite_archivos is not None:
        csv_files = csv_files[:limite_archivos]
# --- FIN DE LÓGICA DE FILTRADO ---

print(f"--- SISTEMA DE PROCESAMIENTO Y GRAFICACIÓN ({periodo_elegido.upper()}) ---")
print(f"Archivos encontrados tras aplicar filtros: {len(csv_files)}")
print("-" * 60)

for archivo in csv_files:
    try:
        nombre_archivo = os.path.basename(archivo)
        
        # Generar el nuevo nombre del archivo quitando '_corregido' y añadiendo la escala
        nombre_base = nombre_archivo.replace('.csv', '').replace('', '').strip()
        
        # Leer encabezados para verificar si la variable existe
        encabezados = pd.read_csv(archivo, skiprows=filas_a_saltar, nrows=0).columns
        if variable_a_analizar not in encabezados or 'Fecha' not in encabezados:
            print(f"✗ Saltando {nombre_archivo}: No contiene la variable seleccionada.")
            continue

        # Cargar los datos
        df = pd.read_csv(archivo, skiprows=filas_a_saltar, usecols=['Fecha', variable_a_analizar])
        df = df.rename(columns={variable_a_analizar: 'Valor'})
        df['Fecha'] = pd.to_datetime(df['Fecha'], format='%d/%m/%Y', errors='coerce')
        df['Valor'] = pd.to_numeric(df['Valor'], errors='coerce')
        df = df.dropna(subset=['Fecha', 'Valor'])
        
        # Establecer la fecha como índice para permitir el agrupamiento temporal (resample)
        df.set_index('Fecha', inplace=True)
        
        # Agrupar los datos según la variable (Suma para precipitación, Promedio para lo demás)
        es_precipitacion = "precipitaci" in variable_a_analizar.lower()
        if es_precipitacion:
            df_agrupado = df.resample(regla_resample)['Valor'].sum().reset_index()
            tipo_acumulacion = "Acumulada"
        else:
            df_agrupado = df.resample(regla_resample)['Valor'].mean().reset_index()
            tipo_acumulacion = "Promedio"

        eje_x = df_agrupado['Fecha'].dt.strftime(formato_fecha).tolist()
        posiciones_fijas = list(range(len(eje_x)))
        valores_y = df_agrupado['Valor'].tolist()

        nombre_limpio = nombre_archivo[11:].strip().replace("_corregido.csv", "").replace(".csv", "") if len(nombre_archivo) > 11 else nombre_base
        var_corta = "PPT" if es_precipitacion else "VAR"

        # ==========================================
        # GUARDADO DE RESULTADOS
        # ==========================================

        # --- GESTIÓN DE CARPETAS ---
        output_folder = os.path.join(ruta_base_salida, f"Resultados_{var_corta}_{periodo_elegido}")
        if not os.path.exists(output_folder):
            os.makedirs(output_folder)

        carpeta_estacion = os.path.join(output_folder, nombre_base)
        os.makedirs(carpeta_estacion, exist_ok=True)
        
        output_name = os.path.join(carpeta_estacion, f"{nombre_base}_{var_corta}.xlsx") 

        # --- PREPARACIÓN DE TABLA PARA EXCEL ---
        # Hacemos una copia descartando la columna auxiliar de la gráfica (ya manejado al guardar directo)
        
        # --- GENERACIÓN DEL EXCEL ---
        with pd.ExcelWriter(output_name, engine='openpyxl') as writer:
            df_agrupado.to_excel(writer, sheet_name=f'{var_corta} {periodo_elegido}', index=False)

        # Gráfico Único (Se adapta a cualquier periodo)
        plt.figure(figsize=(12, 4))
        
        if es_precipitacion:
            # Agregamos borde negro fino a las barras con un tono azul para precipitación
            plt.bar(posiciones_fijas, valores_y, color='#2b8cbe', edgecolor='black', linewidth=0.4)
        else:
            plt.plot(posiciones_fijas, valores_y, marker='o', color='#d62728', linestyle='-')

        # Configuración estética y labels
        plt.xlabel(label_eje_x, fontsize=15)
        plt.title(f'{variable_a_analizar.split()[0]} {tipo_acumulacion} {periodo_elegido}\n Estacion {nombre_limpio}', fontsize=14)
        plt.ylabel(variable_a_analizar, fontsize=15)

        # Margen dinámico usado para eje X y Y
        max_val = max(valores_y) if valores_y else 1
        max_val_margin = max_val * 1.15
        plt.ylim(0, max_val_margin)
        plt.xlim(-2, len(posiciones_fijas) + 1)

        #Colocamos fecha de creacion como marca de agua
        now = dt.datetime.now().strftime('%d/%m/%Y %I:%M %p') #Creamos una hora para la hoja de salida
        plt.text(1, -0.27, f'Fecha de creación: {now}', 
                 fontsize=10, ha='right', va='bottom', 
                 color='gray', alpha=0.1, transform=plt.gca().transAxes) #Ingresamos la hora de creacion en la hoja

        # Evitamos la superposicion de etiquetas en el eje x
        ax = plt.gca()
        
        # 1. Determinamos el intervalo dinámico
        total_datos = len(posiciones_fijas)
        
        # Dibuja todas las barras, pero solo imprime ~75 textos de fecha espaciados
        if total_datos > 75:
            intervalo = total_datos // 75
        else:
            intervalo = 1

        # 2. Seleccionamos las posiciones donde se imprimirá el texto
        ticks_seleccionados = posiciones_fijas[::intervalo]
        
        # FixedLocator asegura que NO existan duplicados y se alineen al centro de la barra
        ax.xaxis.set_major_locator(ticker.FixedLocator(ticks_seleccionados))

        # 3. Formateador que extrae el texto de tu lista eje_x
        @ticker.FuncFormatter
        def format_date(x, pos):
            idx = int(round(x))
            if 0 <= idx < len(eje_x):
                return eje_x[idx]
            return ''
            
        ax.xaxis.set_major_formatter(format_date)

        # 4. Ajustes visuales finales
        # 'ha=center' con rotation=90 alinea el texto con el centro exacto del tick/barra
        plt.xticks(rotation=90, ha='center', va='top', fontsize=10)
        
        # Eliminamos el padding extra para que queden bien pegadas a la base
        ax.tick_params(axis='x', which='major', pad=2)

        # Ajustar los márgenes del dibujo para dar más aire abajo
        plt.subplots_adjust(right=0.9, bottom=0.25)
        plt.tight_layout()
        
        # --- GUARDAR IMAGEN ---
        ruta_grafica = os.path.join(carpeta_estacion, f"{nombre_base}_{var_corta}.png")
        plt.savefig(ruta_grafica, dpi=800, bbox_inches='tight')
        plt.close() 

        print(f"✅ Archivos guardados exitosamente en: {carpeta_estacion}")

    except Exception as e:
        print(f"❌ Error procesando {nombre_archivo}: {e}")

print("-" * 60)
print("Proceso finalizado.")

--- SISTEMA DE PROCESAMIENTO Y GRAFICACIÓN (DIARIO) ---
Archivos encontrados tras aplicar filtros: 4
------------------------------------------------------------


C:\Users\ricar\AppData\Local\Temp\ipykernel_32176\2983862165.py:102: DtypeWarning: Columns (4: Precipitación [mm]) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(archivo, skiprows=filas_a_saltar, usecols=['Fecha', variable_a_analizar])


✅ Archivos guardados exitosamente en: C:\Users\ricar\OneDrive\Almacen\Trabajo\5. SIMAC\9. Indicadores_climaticos\4.Revision_de_series\7.Agrupacion de series\4.Resultados\Resultados_PPT_Diario\001 - 016 - Q. Tesorito


C:\Users\ricar\AppData\Local\Temp\ipykernel_32176\2983862165.py:102: DtypeWarning: Columns (4: Precipitación [mm]) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(archivo, skiprows=filas_a_saltar, usecols=['Fecha', variable_a_analizar])


✅ Archivos guardados exitosamente en: C:\Users\ricar\OneDrive\Almacen\Trabajo\5. SIMAC\9. Indicadores_climaticos\4.Revision_de_series\7.Agrupacion de series\4.Resultados\Resultados_PPT_Diario\001 - 016 - Q. Tesorito2
✅ Archivos guardados exitosamente en: C:\Users\ricar\OneDrive\Almacen\Trabajo\5. SIMAC\9. Indicadores_climaticos\4.Revision_de_series\7.Agrupacion de series\4.Resultados\Resultados_PPT_Diario\001 - 016 - Q. Tesorito_corregido
✅ Archivos guardados exitosamente en: C:\Users\ricar\OneDrive\Almacen\Trabajo\5. SIMAC\9. Indicadores_climaticos\4.Revision_de_series\7.Agrupacion de series\4.Resultados\Resultados_PPT_Diario\001 - 016 - Q. Tesorito_corregido2
------------------------------------------------------------
Proceso finalizado.


In [19]:
# =============================================================================
# MÓDULO DE GRAFICACIÓN CON FILTRO DE PERIODO ESPECÍFICO
# =============================================================================
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import datetime as dt

# =============================================================================
# 1. VARIABLES PARA SELECCIONAR EL PERIODO A VISUALIZAR
# =============================================================================
fecha_inicio = "2025-01-01" # Modifica esta fecha (Formato AAAA-MM-DD)
fecha_fin = "2025-12-31"    # Modifica esta fecha (Formato AAAA-MM-DD)

# NOTA: Este bloque asume que las variables 'df_agrupado', 'regla', 
# 'escala_agrupacion', 'variable_a_analizar', 'tipo_agrupacion' y 'nombre_limpio' 
# ya están cargadas en la memoria de tu Jupyter Notebook de la celda anterior.

# Convertir las cadenas de texto a formato datetime
inicio = pd.to_datetime(fecha_inicio)
fin = pd.to_datetime(fecha_fin)

# Filtrar el DataFrame agrupado por las fechas seleccionadas
mask = (df_agrupado['Fecha'] >= inicio) & (df_agrupado['Fecha'] <= fin)
df_filtrado = df_agrupado.loc[mask].copy()

if df_filtrado.empty:
    print(f"⚠️ No hay datos disponibles para el periodo entre {fecha_inicio} y {fecha_fin}.")
else:
    # =============================================================================
    # 2. CONFIGURACIÓN GRÁFICA DEL PERIODO FILTRADO
    # =============================================================================
    eje_x = [formatear_etiqueta_fecha(f, escala_agrupacion) for f in df_filtrado['Fecha']]
    alturas_val = df_filtrado['Valor'].fillna(0).tolist()
    posiciones_fijas = list(range(len(alturas_val)))
    
    plt.figure(figsize=(12, 4), dpi=800)
    
    # Borde negro fino con tono azul (estilo original)
    plt.plot(posiciones_fijas, alturas_val, color='#2b8cbe', linewidth=0.4)

    # Configuración estética y labels
    plt.xlabel(regla['label_x'], fontsize=15)
    plt.ylabel(variable_a_analizar, fontsize=15)
    
    # Se actualiza el título para reflejar el periodo seleccionado
    titulo = f"{variable_a_analizar.split(' ')[0]} {tipo_agrupacion} {escala_agrupacion.capitalize()}\n Estación {nombre_limpio}\n(Periodo: {fecha_inicio} a {fecha_fin})"
    plt.title(titulo, fontsize=14)

    # Margen dinámico para el eje Y
    max_val = max(alturas_val) if alturas_val else 1
    plt.ylim(0, max_val * 1.15)
    plt.xlim(-2, len(posiciones_fijas) + 1)

    # Marca de agua de creación
    now = dt.datetime.now().strftime('%d/%m/%Y %I:%M %p')
    plt.text(1, -0.27, f'Fecha de creación: {now}', 
             fontsize=10, ha='right', va='bottom', 
             color='gray', alpha=0.5, transform=plt.gca().transAxes)

    # Lógica para evitar la superposición de etiquetas en el eje X
    ax = plt.gca()
    total_datos = len(posiciones_fijas)
    intervalo = total_datos // 75 if total_datos > 75 else 1
    ticks_seleccionados = posiciones_fijas[::intervalo]
    
    ax.xaxis.set_major_locator(ticker.FixedLocator(ticks_seleccionados))

    @ticker.FuncFormatter
    def format_date(x, pos):
        idx = int(round(x))
        if 0 <= idx < len(eje_x):
            return eje_x[idx]
        return ''
    
    ax.xaxis.set_major_formatter(format_date)

    # Ajustes visuales finales de rotación y padding
    plt.xticks(rotation=90, ha='center', va='top', fontsize=regla['font'])
    ax.tick_params(axis='x', which='major', pad=2)
    plt.subplots_adjust(right=0.9, bottom=0.25)
    plt.tight_layout()

    # Mostrar la gráfica interactiva en el notebook
    plt.show()

NameError: name 'formatear_etiqueta_fecha' is not defined